[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Breeze-TTS-2_Colab.ipynb) [View on GitHub](https://github.com/Skquark/AEI-Colab-Notebooks/blob/main/Breeze-TTS-2_Colab.ipynb)

**Open in Colab** ↑ for one-click run · **View on GitHub** ↑ for source


# 🎙️ Breeze TTS 2 — Voice Design, Clone & Direction (EN/ZH, 3B)

> **Runtime required:** GPU. Works on **T4 (16 GB)** — eager inference needs ≈7.7 GiB VRAM.
> Recommended: **L4 (24 GB)** or **A100** for faster generation.

A Colab port of [BreezeBlue/Breeze-TTS-2](https://huggingface.co/BreezeBlue/Breeze-TTS-2) —
an open-weight bilingual (English/Chinese) TTS model ranked **#1 among open-weight models**
on the Artificial Analysis TTS leaderboard. Three modes:

- **Voice Design** — create a voice from a natural-language description (no reference audio).
- **Voice Clone** — clone a speaker from clean reference audio + its exact transcript.
- **Voice Direction** — clone a reference voice while steering tone, emotion, and pace.

Supports inline vocal events in the text: `(laugh)`, `(sigh)`, `(cough)`, `(clears throat)`
(English); `[笑]`, `[叹气]`, `[咳嗽]`, `[清嗓子]` (Chinese).

## How it works

Breeze TTS 2 is a 3-stage LM-style speech generator:

1. **Backbone** (Qwen3-1B, 28 layers) — autoregressively predicts the first semantic
   codebook token per 12.5 Hz frame, conditioned on the text + voice description.
2. **Depth decoder** (12-layer llama-100M) — fills in the remaining 15 of 16 RVQ codebooks
   per frame.
3. **Codec** (Kyutai Mimi, 32 quantizers @ 12.5 Hz) — decodes the frame tokens into
   **24 kHz mono** audio.

```
text + voice instruction ──► Qwen3 backbone (AR) ──► depth decoder (16 codebooks)
                                                            │
                                                    Mimi codec (12.5 Hz frames)
                                                            │
                                                            ▼
                                                    24 kHz mono WAV
```

## Modes

| Mode | Inputs | CFG scale | Use for |
|------|--------|-----------|---------|
| **Voice Design** | text + voice description | 4.0 (recommended) | Fresh narrator voices, no reference needed |
| **Voice Clone** | text + ref audio + exact transcript | 1.0 | Your own voice, or a licensed sample |
| **Voice Direction** | text + ref audio + transcript + direction | 4.0 (recommended) | Clone + steer emotion/pace per scene |

For **audiobook narration**, Voice Direction is the killer combo: clone your voice once,
then per-paragraph directions like *"speak slowly with a restrained, serious tone"*.
Inline `(sigh)` / `(laugh)` events add the human touches mid-sentence.

## ⚠️ License

- **Code** (github.com/breezeblue-ai/breeze-tts): Apache 2.0.
- **Weights & outputs**: BreezeBlue Research and **Non-Commercial** License.
  Commercial use requires written authorization from RESONIA, INC. (contact@breeze.blue).
  Personal / research / indie audiobook projects are fine; commercial audiobooks need a license.

## Quick start

1. **Runtime → Change runtime type → GPU** (T4 works; L4/A100 faster)
2. Run **STEP 1** — clones the breeze-tts repo + installs deps. ≈2-4 min.
3. Run **STEP 2** — downloads the ≈7.2 GB checkpoint to Drive cache. First run ≈5-10 min.
4. Run **STEP 3** — imports + lazy pipe loader + `generate_speech()`.
5. Run **STEP 4** for the tabbed Gradio UI, **STEP 6** for a quick test, or **STEP 7** for batch.
6. **STEP 5** keep-alive prevents Colab disconnects.
7. **STEP 7 chapter mode** — paste a chapter's plain text, get one concatenated WAV +
   per-paragraph WAVs. Perfect for audiobook narration.

## Memory

| GPU | VRAM | Works? | Notes |
|-----|------|--------|-------|
| **A100 80/40 GB** | 40+ GB | Yes | Fastest. |
| **L4 24 GB** | 24 GB | Yes | Recommended. |
| **T4 16 GB** | 16 GB | Yes | ≈7.7 GiB eager fits with room to spare. |

## Outputs

Each generation produces a **24 kHz mono WAV** (`*.wav` in `AEI_3D_Out/Breeze-TTS-2/`),
written as PCM_16. Batch mode also writes a `batch_log.jsonl` resume log.


In [ ]:
#@title STEP 1 — Install breeze-tts repo + dependencies (Drive-persistent)

"""
• Clones github.com/breezeblue-ai/breeze-tts (Apache 2.0 inference code) to Drive
• Installs transformers 4.57.3 + qwen-tts 0.1.1 (pinned per the official Space)
• Installs soundfile / librosa / einops for audio IO
• Verifies GPU + prints VRAM
"""
import os, sys, time, subprocess, pathlib

print('=' * 72)
print('Breeze TTS 2 — Install + Setup')
print('=' * 72)

try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / 1024**3:.1f} GB)')
    else:
        raise SystemExit('No GPU detected. Runtime → Change runtime type → GPU (T4 or better).')
except ImportError:
    raise SystemExit('torch missing — connect a GPU runtime first.')

# ── Mount Drive for persistent cache ──
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

REPO_DIR = pathlib.Path('/content/drive/MyDrive/AEI_ComfyUI/breeze-tts')
if not REPO_DIR.exists():
    print(f'  Cloning breezeblue-ai/breeze-tts -> {REPO_DIR} ...')
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/breezeblue-ai/breeze-tts.git',
                    str(REPO_DIR)], check=True)
else:
    print(f'  breeze-tts repo already at {REPO_DIR}')
sys.path.insert(0, str(REPO_DIR))

OUT_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/Breeze-TTS-2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Install pinned deps ──
t0 = time.time()
print('\n[1/2] Installing pinned deps (transformers, qwen-tts, accelerate) ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.57.3', 'qwen-tts==0.1.1', 'accelerate==1.12.0',
                'soundfile>=0.13', 'numpy>=2.0', 'librosa>=1.0', 'einops',
                'gradio==5.50.0', 'tqdm'],
               check=False)
print(f'  deps ready in {time.time()-t0:.1f}s')

# ── Verify imports ──
print('\nVerifying imports ...')
try:
    import transformers
    print(f'  transformers : {transformers.__version__}')
except Exception as e:
    print(f'  [FAIL] transformers: {e}')
    raise
try:
    from qwen_tts import Qwen3TTSTokenizer
    print('  qwen_tts     : OK (Qwen3TTSTokenizer importable)')
except Exception as e:
    print(f'  [FAIL] qwen_tts: {e}')
    raise
try:
    from breeze_infer.runtime import load_runtime
    from breeze_infer.templates import get_template, prepare_inputs
    from models.fast_streaming import FastBreezeStreamingRuntime, FastStreamingConfig
    print('  breeze_infer : OK (runtime + templates + fast_streaming)')
except Exception as e:
    print(f'  [FAIL] breeze_infer (is sys.path set?): {e}')
    raise

print('\n' + '=' * 72)
print('STEP 1 done. Next: run STEP 2 to download weights (~7.2 GB).')
print('=' * 72)


In [ ]:
#@title STEP 2 — Download Breeze-TTS-2 weights to Drive cache

"""
Downloads the 3B checkpoint (~6.5 GB in 2 safetensors shards) plus the
bundled Mimi audio tokenizer (~0.64 GB) via snapshot_download.
Total: ~7.2 GB on Drive. Cached; subsequent runs reuse the cache.

First run: ~5-10 min depending on Drive FUSE throughput. Subsequent: 0 s.
The cache lives at <HF_HOME>/hub/models--BreezeBlue--Breeze-TTS-2/.
"""
import os, sys, time, pathlib
from huggingface_hub import snapshot_download

cache_root_str = os.environ.get(
    'HF_HOME', '/content/drive/MyDrive/AEI_3D_Cache/Breeze-TTS-2/huggingface')
cache_root = pathlib.Path(cache_root_str)
cache_root.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('HF_HOME', str(cache_root))
os.environ.setdefault('HUGGINGFACE_HUB_CACHE', str(cache_root))

OUT_DIR = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/Breeze-TTS-2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_ID = 'BreezeBlue/Breeze-TTS-2'
ALLOW_PATTERNS = [
    'config.json',
    'generation_config.json',
    'model-*.safetensors',
    'model.safetensors.index.json',
    'special_tokens_map.json',
    'tokenizer.json',
    'tokenizer_config.json',
    'audio_tokenizer/**',
    'LICENSE', 'README.md',
]

print('=' * 72)
print(f'Downloading {REPO_ID} to {cache_root}/hub/ ...')
print(f'Allow patterns: {len(ALLOW_PATTERNS)} entries')
print('=' * 72)

t0 = time.time()
try:
    local_path = snapshot_download(
        repo_id=REPO_ID,
        cache_dir=str(cache_root),
        allow_patterns=ALLOW_PATTERNS,
        max_workers=4,
    )
    elapsed = time.time() - t0
    print(f'\nDone in {elapsed/60:.1f} min. Local path: {local_path}')

    total_bytes = 0
    for root, dirs, files in os.walk(local_path):
        for f in files:
            fp = os.path.join(root, f)
            total_bytes += os.path.getsize(fp)
    print(f'Total on disk: {total_bytes / 1024**3:.2f} GB')

    print('\nComponents present:')
    for entry in sorted(os.listdir(local_path)):
        full = os.path.join(local_path, entry)
        if os.path.isdir(full):
            sz = sum(os.path.getsize(os.path.join(r, f2))
                     for r, _, files in os.walk(full) for f2 in files)
            print(f'  {entry.ljust(30)} {sz/1024**3:6.2f} GB')
        else:
            print(f'  {entry.ljust(30)} (file, {os.path.getsize(full)/1024:.1f} KB)')

    print('\n' + '=' * 72)
    print('STEP 2 done. Next: run STEP 3 to register the lazy loader.')
    print('=' * 72)
except Exception as e:
    print(f'\n[FAIL] download error: {type(e).__name__}: {e}')
    print('Possible causes:')
    print('  1. No internet connection')
    print(f'  2. {REPO_ID} unreachable (retry once or twice)')
    print('  3. Drive not mounted (re-run STEP 1)')
    raise


In [ ]:
#@title STEP 3 — Imports, lazy pipe loader, generate_speech()

"""
Imports + a single generate_speech() function that wraps the Breeze TTS 2
runtime. The pipe loads lazily on first call (~30-60 s init) and is cached
in a module-level global.

Three modes (mirroring the official Space):
  - 'design'    — voice from natural-language description. CFG 4.0 recommended.
  - 'clone'     — voice from ref audio + exact transcript. CFG 1.0.
  - 'direction' — clone + steer tone/emotion/pace via instruction. CFG 4.0.

Output: 24 kHz mono WAV (PCM_16), saved to OUT_DIR.

What is exposed to other cells via builtins:
  - _generate_speech(text, instruction, mode, ref_audio, ref_text, cfg_scale, seed, ...)
  - _load_runtime()   (re-runs the loader, e.g. after OOM recovery)
  - _save_wav(...)    (WAV writer for manual use)
  - Breeze_TTS_OUT_DIR (output path)
"""
import os, sys, time, json, gc, random, builtins, pathlib
from pathlib import Path

import numpy as np
import torch
import soundfile as sf

# breeze-tts repo was cloned to Drive in STEP 1
REPO_DIR = '/content/drive/MyDrive/AEI_ComfyUI/breeze-tts'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

_cache_root_str = os.environ.get(
    'HF_HOME', '/content/drive/MyDrive/AEI_3D_Cache/Breeze-TTS-2/huggingface')
OUT_DIR = Path('/content/drive/MyDrive/AEI_3D_Out/Breeze-TTS-2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'BreezeBlue/Breeze-TTS-2'

# Lazy runtime state
RUNTIME = None
MODEL = None
TOKENIZER = None
AUDIO_TOKENIZER = None

MODE_CFG_DEFAULTS = {'design': 4.0, 'clone': 1.0, 'direction': 4.0}


def _get_repo_dir():
    """Find the local snapshot dir for the Breeze-TTS-2 checkpoint."""
    cache_root = Path(_cache_root_str)
    hub_root = cache_root / 'hub' / 'models--BreezeBlue--Breeze-TTS-2' / 'snapshots'
    if hub_root.exists():
        snaps = sorted(hub_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
        if snaps:
            return str(snaps[0])
    if (cache_root / 'config.json').exists():
        return str(cache_root)
    raise FileNotFoundError(
        f'Breeze-TTS-2 checkpoint not found at {cache_root}. Re-run STEP 2.')


def _load_runtime(force_reload=False):
    """Lazy runtime loader. Caches globally so subsequent calls don't reload."""
    global RUNTIME, MODEL, TOKENIZER, AUDIO_TOKENIZER
    if RUNTIME is not None and not force_reload:
        return RUNTIME, MODEL, TOKENIZER, AUDIO_TOKENIZER

    from breeze_infer.runtime import (
        load_runtime as _lr,
        update_generation_config_for_breeze,
    )
    from breeze_infer.templates import get_template, prepare_inputs
    from models.fast_streaming import FastBreezeStreamingRuntime, FastStreamingConfig

    # publish to builtins for STEP 6/7
    builtins._breeze_get_template = get_template
    builtins._breeze_prepare_inputs = prepare_inputs

    repo_dir = _get_repo_dir()
    print(f'\nLoading Breeze TTS 2 from {repo_dir} ...')
    t0 = time.time()

    tokenizer, model, audio_tokenizer = _load_runtime_impl(repo_dir)
    update_generation_config_for_breeze(model)

    config = FastStreamingConfig(
        max_new_tokens=1500,
        max_seq_len=2048,
        fast_all=None,
        repetition_penalty=1.1,
    )
    runtime = FastBreezeStreamingRuntime(
        model, audio_tokenizer, config, tokenizer=tokenizer)

    elapsed = time.time() - t0
    print(f'  Runtime ready in {elapsed:.1f}s  (sample_rate={runtime.sample_rate})')

    RUNTIME = runtime
    MODEL = model
    TOKENIZER = tokenizer
    AUDIO_TOKENIZER = audio_tokenizer
    return runtime, model, tokenizer, audio_tokenizer


def _load_runtime_impl(repo_dir):
    """Thin wrapper so OOM recovery can swap device without touching globals."""
    from breeze_infer.runtime import load_runtime as _lr
    return _lr(Path(repo_dir), device='cuda', attn_implementation='eager')


def _slug(text, max_len=40):
    import re
    s = re.sub(r'[^a-zA-Z0-9_\-]+', '-', text or '').strip('-')
    return s[:max_len] or 'speech'


def _save_wav(audio_np, sample_rate, out_path):
    """Save a float32 mono array as 16-bit PCM WAV."""
    if audio_np.ndim > 1:
        audio_np = audio_np.squeeze()
    audio_np = np.clip(audio_np, -1.0, 1.0)
    sf.write(str(out_path), audio_np, sample_rate, subtype='PCM_16')
    return out_path


def generate_speech(text, instruction='Speak clearly and naturally.',
                    mode='design', ref_audio_path=None, ref_text=None,
                    cfg_scale=None, seed=42, save=True, prefix='speech'):
    """Generate one speech clip and save it as a WAV file.

    Args:
        text: text to speak. Supports inline vocal events:
              (laugh), (sigh), (cough), (clears throat) in English;
              [笑], [叹气], [咳嗽], [清嗓子] in Chinese.
        instruction: voice description (mode='design') or direction
                     (mode='direction'). Ignored for mode='clone'.
        mode: 'design' | 'clone' | 'direction'
        ref_audio_path: path to reference audio (clone/direction only).
        ref_text: exact transcript of the reference audio (clone/direction only).
        cfg_scale: classifier-free guidance scale. None = mode default
                   (design 4.0, clone 1.0, direction 4.0).
        seed: random seed for reproducibility.
        save: write WAV to OUT_DIR.
        prefix: filename prefix.

    Returns: dict with 'wav_path', 'sample_rate', 'duration_s', 'seed_used'.
    """
    runtime, model, tokenizer, audio_tokenizer = _load_runtime()
    from breeze_infer.runtime import set_all_seeds

    text = (text or '').strip()
    if not text:
        raise ValueError('Text to speak is required.')

    mode = (mode or 'design').strip().lower()
    if mode not in ('design', 'clone', 'direction'):
        raise ValueError(f"mode must be 'design'|'clone'|'direction', got {mode!r}")

    if cfg_scale is None:
        cfg_scale = MODE_CFG_DEFAULTS[mode]

    has_ref = ref_audio_path is not None and ref_text and str(ref_text).strip()
    if mode in ('clone', 'direction') and not has_ref:
        raise ValueError(
            f"mode={mode!r} needs both ref_audio_path and ref_text "
            '(the exact transcript of the reference audio).')
    if mode == 'design' and has_ref:
        print('  [warn] mode=design ignores ref_audio; switch mode=direction to use it.')

    seed_used = int(seed)
    if seed_used <= 0:
        seed_used = random.randint(1, 2**31 - 1)
    set_all_seeds(seed_used)

    if mode == 'design':
        instruction = instruction or 'Speak clearly and naturally.'
    elif mode == 'clone':
        instruction = 'Speak clearly and naturally.'
    else:  # direction: the instruction IS the steering input
        if not (instruction and str(instruction).strip()):
            raise ValueError("mode='direction' needs an instruction "
                             '(e.g. Speak slowly with a restrained, serious tone.)')
        instruction = str(instruction).strip()

    request = {
        'id': f'colab-{int(time.time()*1000)}',
        'text': text,
        'instruction': instruction,
        'speaker': 'S0',
    }
    template_name = 'tts_instruction'
    if has_ref:
        request['ref_audio_path'] = str(ref_audio_path)
        request['ref_text'] = str(ref_text).strip()
        template_name = 'ref_edit_tata'

    from breeze_infer.templates import get_template, prepare_inputs
    inputs = prepare_inputs(
        tokenizer, AUDIO_TOKENIZER, model, [request],
        get_template(template_name),
        guidance_scale=float(cfg_scale),
        guidance_scale_ref=None,
        guidance_scale_ins=None,
    )

    print(f'\nGenerating ({mode}, cfg={cfg_scale}, seed={seed_used}) ...')
    t0 = time.time()
    audio_chunks = []
    for chunk in runtime.iter_audio_chunks(inputs, request_id=request['id']):
        audio_chunks.append(chunk.audio)
    if not audio_chunks:
        raise RuntimeError('Model produced no audio output.')
    audio = np.concatenate(audio_chunks, axis=0)
    sr = runtime.sample_rate
    elapsed = time.time() - t0
    duration_s = audio.shape[-1] / sr
    print(f'  Generated {duration_s:.1f}s of audio in {elapsed:.1f}s '
          f'({duration_s/elapsed:.2f}x realtime)')

    result = {
        'duration_s': duration_s,
        'sample_rate': sr,
        'seed_used': seed_used,
        'mode': mode,
        'cfg_scale': cfg_scale,
        'wav_path': None,
    }
    if save:
        slug = _slug(text[:30])
        ts = int(time.time())
        out_path = OUT_DIR / f'{prefix}_{slug}_d{int(duration_s)}s_cfg{cfg_scale}_seed{seed_used}_{ts}.wav'
        _save_wav(audio, sr, out_path)
        result['wav_path'] = str(out_path)
        print(f'  Saved: {out_path}')

    return result


# Expose to builtins for STEP 4/6/7
builtins._generate_speech = generate_speech
builtins._load_breeze_runtime = _load_runtime
builtins._save_breeze_wav = _save_wav
builtins.Breeze_TTS_OUT_DIR = OUT_DIR

print('STEP 3 ready. Runtime loads lazily on first generate_speech() call.')
print('Run STEP 4 (Gradio), STEP 6 (quick test), or STEP 7 (batch) next.')


In [ ]:
#@title STEP 4 — (Optional) Gradio UI for Breeze TTS 2

"""
Tabbed UI mirroring the official Space: Voice Design / Voice Clone /
Voice Direction. Clone + Direction auto-transcribe reference audio with
CPU Whisper (openai/whisper-small) so you don't have to type the transcript.
"""
import os, time, random, builtins, tempfile, threading
from pathlib import Path

import gradio as gr
import numpy as np
import soundfile as sf

_generate_speech = builtins._generate_speech
OUT_DIR = builtins.Breeze_TTS_OUT_DIR

ASR_MODEL_ID = 'openai/whisper-small'
_asr_pipe = None
_asr_lock = threading.Lock()


def _get_asr():
    global _asr_pipe
    with _asr_lock:
        if _asr_pipe is None:
            from transformers import pipeline
            _asr_pipe = pipeline(
                'automatic-speech-recognition',
                model=ASR_MODEL_ID,
                device='cpu',
                chunk_length_s=30,
            )
    return _asr_pipe


def transcribe_reference(ref_audio_path):
    """Auto-transcribe uploaded/recorded reference audio with CPU Whisper."""
    if not ref_audio_path:
        return gr.update()
    try:
        asr = _get_asr()
    except Exception as exc:
        raise gr.Error(f'Could not load the transcription model: {exc}') from exc
    try:
        import librosa
        audio, _ = librosa.load(ref_audio_path, sr=16000, mono=True)
        asr_input = {'raw': audio, 'sampling_rate': 16000}
    except Exception:
        asr_input = str(ref_audio_path)
    try:
        result = asr(asr_input, generate_kwargs={'task': 'transcribe'})
    except Exception as exc:
        raise gr.Error(f'Automatic transcription failed: {exc}') from exc
    text = (result.get('text') or '').strip()
    if not text:
        raise gr.Error('Automatic transcription produced no text — type the transcript manually.')
    return text


def _run_design(text, instruction, cfg_scale, seed):
    if not text.strip():
        raise gr.Error('Text to speak is required.')
    if seed is None or int(seed) <= 0:
        seed = random.randint(1, 2**31 - 1)
    try:
        result = _generate_speech(
            text=text, instruction=instruction, mode='design',
            cfg_scale=float(cfg_scale), seed=int(seed), save=True, prefix='design')
        return result['wav_path'], result['seed_used'], f"{result['duration_s']:.1f}s"
    except Exception as e:
        raise gr.Error(f'{type(e).__name__}: {e}') from e


def _run_clone(text, ref_audio, ref_text, seed):
    if not text.strip():
        raise gr.Error('Text to speak is required.')
    if ref_audio is None:
        raise gr.Error('Reference audio is required for Voice Clone.')
    if not (ref_text and ref_text.strip()):
        raise gr.Error('Reference transcript is required (exact words spoken in the ref audio).')
    if seed is None or int(seed) <= 0:
        seed = random.randint(1, 2**31 - 1)
    try:
        result = _generate_speech(
            text=text, mode='clone', ref_audio_path=ref_audio, ref_text=ref_text,
            seed=int(seed), save=True, prefix='clone')
        return result['wav_path'], result['seed_used'], f"{result['duration_s']:.1f}s"
    except Exception as e:
        raise gr.Error(f'{type(e).__name__}: {e}') from e


def _run_direction(text, instruction, ref_audio, ref_text, cfg_scale, seed):
    if not text.strip():
        raise gr.Error('Text to speak is required.')
    if ref_audio is None:
        raise gr.Error('Reference audio is required for Voice Direction.')
    if not (ref_text and ref_text.strip()):
        raise gr.Error('Reference transcript is required.')
    if seed is None or int(seed) <= 0:
        seed = random.randint(1, 2**31 - 1)
    try:
        result = _generate_speech(
            text=text, instruction=instruction, mode='direction',
            ref_audio_path=ref_audio, ref_text=ref_text,
            cfg_scale=float(cfg_scale), seed=int(seed), save=True, prefix='direction')
        return result['wav_path'], result['seed_used'], f"{result['duration_s']:.1f}s"
    except Exception as e:
        raise gr.Error(f'{type(e).__name__}: {e}') from e


with gr.Blocks(title='Breeze TTS 2', theme=gr.themes.Citrus()) as demo:
    gr.Markdown(
        '# 🎙️ Breeze TTS 2\n'
        'Bilingual (EN/ZH) TTS with **Voice Design**, **Voice Clone**, and '
        '**Voice Direction**. Inline vocal events: `(laugh)`, `(sigh)`, `(cough)`, '
        '`(clears throat)`.'
    )
    with gr.Tabs():
        with gr.Tab('🎨 Voice Design'):
            with gr.Row():
                with gr.Column(scale=3):
                    design_text = gr.Textbox(
                        label='Text to speak', lines=3,
                        placeholder='Welcome aboard. Your journey begins now.',
                        info='Inline vocal events: (laugh), (sigh), (cough), (clears throat)')
                    design_instruction = gr.Textbox(
                        label='Voice description', lines=2,
                        placeholder='A warm, thoughtful young woman with a clear voice and a calm, reflective delivery.')
                    with gr.Accordion('Advanced settings', open=False):
                        design_cfg = gr.Slider(label='CFG scale', minimum=1.0, maximum=10.0,
                                                value=4.0, step=0.5,
                                                info='Higher = stronger instruction following')
                        design_seed = gr.Number(label='Seed (0 = random)', value=42, precision=0)
                    design_btn = gr.Button('Generate', variant='primary')
                with gr.Column(scale=2):
                    design_output = gr.Audio(label='Generated audio', type='filepath')
                    design_meta = gr.Textbox(label='Seed / duration', interactive=False)

        with gr.Tab('🎙️ Voice Clone'):
            gr.Markdown('Clone a speaker from **clean reference audio** and its **exact transcript**.')
            with gr.Row():
                with gr.Column(scale=3):
                    clone_text = gr.Textbox(
                        label='Text to speak', lines=3,
                        placeholder='It is good to hear your voice again after all this time.')
                    clone_ref_audio = gr.Audio(label='Reference audio', type='filepath')
                    clone_ref_text = gr.Textbox(
                        label='Reference transcript (exact)', lines=2,
                        info='Auto-transcribed with Whisper when you add reference audio — review and edit if needed.',
                        placeholder='This is the exact transcript of the reference audio.')
                    clone_ref_audio.change(
                        fn=transcribe_reference,
                        inputs=[clone_ref_audio], outputs=[clone_ref_text],
                        api_name='transcribe_reference')
                    with gr.Accordion('Advanced settings', open=False):
                        clone_seed = gr.Number(label='Seed (0 = random)', value=42, precision=0)
                    clone_btn = gr.Button('Generate', variant='primary')
                with gr.Column(scale=2):
                    clone_output = gr.Audio(label='Generated audio', type='filepath')
                    clone_meta = gr.Textbox(label='Seed / duration', interactive=False)

        with gr.Tab('🎛️ Voice Direction'):
            gr.Markdown('Clone a reference voice while **steering tone, emotion, and pace**. '
                        'Use CFG 4 for stronger instruction following.')
            with gr.Row():
                with gr.Column(scale=3):
                    dir_text = gr.Textbox(
                        label='Text to speak', lines=3,
                        placeholder='We need to discuss what happened last night.')
                    dir_instruction = gr.Textbox(
                        label='Direction', lines=2,
                        placeholder='Speak slowly with a restrained, serious tone.')
                    dir_ref_audio = gr.Audio(label='Reference audio', type='filepath')
                    dir_ref_text = gr.Textbox(
                        label='Reference transcript (exact)', lines=2,
                        info='Auto-transcribed with Whisper when you add reference audio.',
                        placeholder='This is the exact transcript of the reference audio.')
                    dir_ref_audio.change(
                        fn=transcribe_reference,
                        inputs=[dir_ref_audio], outputs=[dir_ref_text],
                        api_name='transcribe_reference_direction')
                    with gr.Accordion('Advanced settings', open=False):
                        dir_cfg = gr.Slider(label='CFG scale', minimum=1.0, maximum=10.0,
                                             value=4.0, step=0.5,
                                             info='Higher = stronger direction following')
                        dir_seed = gr.Number(label='Seed (0 = random)', value=42, precision=0)
                    dir_btn = gr.Button('Generate', variant='primary')
                with gr.Column(scale=2):
                    dir_output = gr.Audio(label='Generated audio', type='filepath')
                    dir_meta = gr.Textbox(label='Seed / duration', interactive=False)

    def _show_welcome():
        return ('**Breeze TTS 2 ready.** The runtime loads lazily on the first '
                'Generate click (~30-60 s). Use the tabs above to switch modes.')
    demo.load(_show_welcome, inputs=None, outputs=None)

    design_btn.click(fn=_run_design,
                     inputs=[design_text, design_instruction, design_cfg, design_seed],
                     outputs=[design_output, design_meta], api_name='voice_design')
    clone_btn.click(fn=_run_clone,
                    inputs=[clone_text, clone_ref_audio, clone_ref_text, clone_seed],
                    outputs=[clone_output, clone_meta], api_name='voice_clone')
    dir_btn.click(fn=_run_direction,
                  inputs=[dir_text, dir_instruction, dir_ref_audio, dir_ref_text, dir_cfg, dir_seed],
                  outputs=[dir_output, dir_meta], api_name='voice_direction')

from IPython.display import display, clear_output
clear_output()
demo.queue(concurrency_limit=1).launch(share=False, inline=False,
                                        prevent_thread_lock=True, quiet=True)
display(demo)
print('STEP 4 launched. Each generation takes ~5-30 s depending on text length.')


In [ ]:
#@title STEP 5 — Keep Colab alive + session summary

"""
Prevents Colab from disconnecting with a JS heartbeat + prints a summary
of the current session state (GPU, checkpoint, output dir).
"""
import time, os
from pathlib import Path

import IPython.display
display(IPython.display.Javascript("""
function KeepAlive() { console.log('Colab session kept alive at ' + new Date().toISOString()); }
setInterval(KeepAlive, 60000);
"""))

print('=' * 72)
print('Breeze TTS 2 session summary')
print('=' * 72)

try:
    import torch
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'GPU: {p.name} ({p.total_memory / 1024**3:.1f} GB)')
except Exception:
    print('GPU: unavailable')

try:
    import transformers
    print(f'transformers: {transformers.__version__}')
except Exception:
    pass

try:
    repo_dir = builtins._load_breeze_runtime and Path(
        '/content/drive/MyDrive/AEI_3D_Cache/Breeze-TTS-2/huggingface')
    hub_root = repo_dir / 'hub' / 'models--BreezeBlue--Breeze-TTS-2' / 'snapshots'
    if hub_root.exists():
        snaps = sorted(hub_root.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
        if snaps:
            total = sum(os.path.getsize(os.path.join(r, f))
                        for r, _, files in os.walk(snaps[0]) for f in files)
            print(f'Checkpoint: {snaps[0]}')
            print(f'  Total on disk: {total / 1024**3:.2f} GB')
except Exception:
    print('Checkpoint: not found (re-run STEP 2)')

try:
    out_dir = builtins.Breeze_TTS_OUT_DIR
    wavs = sorted(out_dir.glob('*.wav'))
    print(f'Output dir : {out_dir}')
    print(f'  clips: {len(wavs)}')
    for w in wavs[-5:]:
        import soundfile as _sf
        info = _sf.info(str(w))
        print(f'    {info.duration:6.1f}s  {w.name}')
except Exception as e:
    print(f'Output dir: {e}')

print('\nHeartbeat loop (Ctrl+C or Interrupt to stop):')
try:
    while True:
        time.sleep(60)
        print(f'  ... alive at {time.strftime("%H:%M:%S")}')
except KeyboardInterrupt:
    print('Heartbeat stopped.')


In [ ]:
#@title STEP 6 — Quick test (single speech generation)

"""
Generate one clip with form-widget inputs and immediately play it.
MODE: 'design' (voice description), 'clone' (ref audio + transcript),
      'direction' (ref audio + transcript + steering instruction).
"""
import time, random, builtins
from pathlib import Path
from IPython.display import display, Audio, FileLink

TEXT = "Welcome aboard. Your journey begins now. This voice was designed to feel warm and reflective."  #@param {type:'string'}
#@markdown ── Voice description (mode='design' or 'direction') ──
INSTRUCTION = 'A warm, thoughtful narrator with a clear voice and a calm, reflective delivery.'  #@param {type:'string'}
#@markdown ── Voice clone / direction (mode='clone' or 'direction') ──
REF_AUDIO_PATH = ''  #@param {type:'string'}
#@markdown Exact transcript of the reference audio (required for clone/direction).
REF_TEXT = ''  #@param {type:'string'}
MODE = 'design'  #@param ['design', 'clone', 'direction']
CFG_SCALE = 4.0  #@param {type:'slider', min:1.0, max:10.0, step:0.5}
#@markdown CFG 4.0 recommended for design/direction. Clone uses 1.0 regardless.
SEED = 42  #@param {type:'integer'}
SAVE_FILE = True  #@param {type:'boolean'}
PREFIX = 'quicktest'  #@param {type:'string'}

print(f'  Mode   : {MODE}')
print(f'  Text   : {TEXT[:60]}...')
if INSTRUCTION:
    print(f'  Instr  : {INSTRUCTION[:60]}...')
print(f'  CFG    : {CFG_SCALE}, seed={SEED}, save={SAVE_FILE}')
print()

kwargs = {}
if REF_AUDIO_PATH.strip():
    kwargs['ref_audio_path'] = REF_AUDIO_PATH.strip()
if REF_TEXT.strip():
    kwargs['ref_text'] = REF_TEXT.strip()

result = builtins._generate_speech(
    text=TEXT,
    instruction=INSTRUCTION if INSTRUCTION.strip() else None,
    mode=MODE,
    cfg_scale=CFG_SCALE if MODE != 'clone' else 1.0,
    seed=SEED,
    save=SAVE_FILE,
    prefix=PREFIX,
    **kwargs,
)

print(f'  Done: {result["duration_s"]:.1f}s of audio, seed={result["seed_used"]}')
print(f'  File: {result.get("wav_path") or "<preview only - not saved>"}')
print()

if result.get('wav_path'):
    display(Audio(result['wav_path']))
    try:
        from IPython.display import FileLink
        display(FileLink(result['wav_path']))
    except Exception:
        pass


In [ ]:
#@title STEP 7 — Batch generation (JSON scenes OR audiobook chapter mode)

"""
Two batch modes:

1. JSON mode (BATCH_MODE='json'): reads a JSON list of scene dicts, each with
   its own text / instruction / mode / cfg_scale / seed / ref_audio / ref_text.

2. Chapter mode (BATCH_MODE='chapter', for audiobooks): reads a plain-text
   file, splits it into paragraph chunks at blank lines, generates each chunk
   with the same voice settings, then concatenates into ONE long WAV per
   chapter + individual per-paragraph WAVs.

Both modes write a batch_log.jsonl for resume-after-disconnect.
"""
import os, sys, time, json, random, hashlib, builtins
from pathlib import Path
import numpy as np
import soundfile as sf

BATCH_MODE = 'chapter'  #@param ['chapter', 'json']
#@markdown ── Chapter mode (audiobooks) ──
CHAPTER_TEXT_FILE = '/content/drive/MyDrive/AEI_3D_Cache/Breeze-TTS-2/chapter_01.txt'  #@param {type:'string'}
#@markdown One paragraph per block, separated by blank lines. Long paragraphs are
#@markdown split at sentence boundaries if they exceed MAX_CHUNK_CHARS.
MAX_CHUNK_CHARS = 400  #@param {type:'integer'}
#@markdown ── JSON mode ──
BATCH_JSON_PATH = '/content/drive/MyDrive/AEI_3D_Cache/Breeze-TTS-2/batch_speech.json'  #@param {type:'string'}
#@markdown ── Shared voice settings (used by both modes) ──
MODE = 'direction'  #@param ['design', 'clone', 'direction']
INSTRUCTION = 'A warm, clear audiobook narrator with a calm, steady pace.'  #@param {type:'string'}
REF_AUDIO_PATH = ''  #@param {type:'string'}
REF_TEXT = ''  #@param {type:'string'}
CFG_SCALE = 4.0  #@param {type:'slider', min:1.0, max:10.0, step:0.5}
SEED = 42  #@param {type:'integer'}
#@markdown ── Resume / skip ──
SKIP_EXISTING = True  #@param {type:'boolean'}
RESUME_FROM_LOG = True  #@param {type:'boolean'}

OUT_DIR = builtins.Breeze_TTS_OUT_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

_mode_cfg = 1.0 if MODE == 'clone' else CFG_SCALE

# ── Shared helpers ──────────────────────────────────────────────────────
def _slug(text, max_len=40):
    import re
    s = re.sub(r'[^a-zA-Z0-9_\-]+', '-', text or '').strip('-')
    return s[:max_len] or 'clip'


def _split_paragraphs(text, max_chars=400):
    """Split on blank lines; long paragraphs split at sentence boundaries."""
    import re
    paras = [p.strip() for p in text.split('\n\n') if p.strip()]
    out = []
    for p in paras:
        if len(p) <= max_chars:
            out.append(p)
            continue
        sentences = re.split(r'(?<=[.!?])\s+', p)
        cur = ''
        for s in sentences:
            if cur and len(cur) + len(s) + 1 > max_chars:
                out.append(cur.strip())
                cur = s
            else:
                cur = f'{cur} {s}'.strip()
        if cur:
            out.append(cur.strip())
    return out


def _hash_key(text, mode, instruction, cfg, seed):
    h = hashlib.sha256()
    h.update(text.encode('utf-8'))
    h.update(b'\x00')
    h.update(str(mode).encode())
    h.update(b'\x00')
    h.update(str(instruction or '').encode())
    h.update(b'\x00')
    h.update(str(cfg).encode())
    return h.hexdigest()[:16]


_log_path = OUT_DIR / 'batch_speech.log.jsonl'
_completed = {}  # hash -> wav path (from the resume log)
if RESUME_FROM_LOG and _log_path.exists():
    for line in _log_path.read_text().splitlines():
        try:
            entry = json.loads(line)
            if entry.get('status') == 'ok' and entry.get('hash'):
                _completed[entry['hash']] = entry.get('path')
        except Exception:
            pass
    print(f'  Resume log: {len(_completed)} already-completed chunks')

results = []
total_start = time.time()

# ═══ CHAPTER MODE ═══════════════════════════════════════════════════════
if BATCH_MODE == 'chapter':
    chapter_path = Path(CHAPTER_TEXT_FILE)
    if not chapter_path.exists():
        chapter_path.parent.mkdir(parents=True, exist_ok=True)
        starter = ('Chapter One\n\n'
                   'The morning light came slowly to the valley, first touching the '
                   'highest ridge before sliding down the slopes into the trees.\n\n'
                   'It was the kind of light that made everything look new again, as if '
                   'the world had been holding its breath all night and only now '
                   'remembered how to breathe. (sigh) And with that breath, the day began.')
        chapter_path.write_text(starter)
        print(f'  Wrote starter chapter: {chapter_path}')
        print('  Edit it (paragraphs separated by blank lines), then re-run STEP 7.')
        raise SystemExit(0)

    chapter_text = chapter_path.read_text()
    chunks = _split_paragraphs(chapter_text, MAX_CHUNK_CHARS)
    print(f'  Chapter file : {chapter_path.name}')
    print(f'  Paragraphs   : {len(chunks)} chunk(s) after splitting')

    chapter_ts = int(time.time())
    chapter_slug = _slug(chapter_path.stem, 30)
    per_chunk_paths = []
    audio_parts = []
    sr_used = None

    for ci, chunk_text in enumerate(chunks):
        hash_id = _hash_key(chunk_text, MODE, INSTRUCTION, _mode_cfg)

        # Resume: the log stores hash -> wav path, so a previously-completed
        # chunk is picked up by its hash regardless of the timestamped name.
        if SKIP_EXISTING and hash_id in _completed:
            prev_path = _completed[hash_id]
            if prev_path and Path(prev_path).exists():
                print(f'  [{ci+1}/{len(chunks)}] SKIP (resume log): {Path(prev_path).name}')
                per_chunk_paths.append(prev_path)
                continue
            print(f'  [{ci+1}/{len(chunks)}] resume-log entry is stale (file gone); regenerating')

        print(f'  [{ci+1}/{len(chunks)}] {len(chunk_text)} chars, seed={SEED}')
        try:
            kwargs = {}
            if REF_AUDIO_PATH.strip():
                kwargs['ref_audio_path'] = REF_AUDIO_PATH.strip()
            if REF_TEXT.strip():
                kwargs['ref_text'] = REF_TEXT.strip()
            result = builtins._generate_speech(
                text=chunk_text,
                instruction=INSTRUCTION if INSTRUCTION.strip() else None,
                mode=MODE,
                cfg_scale=_mode_cfg,
                seed=SEED,
                save=True,
                prefix=f'chapter_{chapter_slug}_{ci:04d}',
                **kwargs,
            )
            per_chunk_paths.append(result['wav_path'])
            with _log_path.open('a') as f:
                f.write(json.dumps({'mode': 'chapter', 'index': ci, 'status': 'ok',
                                     'path': result['wav_path'],
                                     'duration_s': result['duration_s'],
                                     'seed': result['seed_used'],
                                     'hash': hash_id}) + '\n')
        except Exception as e:
            print(f'  [{ci+1}/{len(chunks)}] FAIL: {type(e).__name__}: {e}')
            with _log_path.open('a') as f:
                f.write(json.dumps({'mode': 'chapter', 'index': ci,
                                     'status': 'fail',
                                     'error': f'{type(e).__name__}: {e}'}) + '\n')

    # Concatenate with a short pause between paragraphs (natural for narration)
    audio_parts = []
    sr_used = None
    for p in per_chunk_paths:
        if p and Path(p).exists():
            audio, sr = sf.read(p)
            if sr_used is None:
                sr_used = sr
            audio_parts.append(audio)
            # 0.35 s of silence between paragraphs
            audio_parts.append(np.zeros(int(0.35 * sr), dtype=audio.dtype))
    if audio_parts:
        audio_parts = audio_parts[:-1]  # drop the trailing pause
    if audio_parts:
        full_audio = np.concatenate(audio_parts, axis=0)
        full_path = OUT_DIR / f'chapter_{chapter_slug}_FULL_seed{SEED}_{chapter_ts}.wav'
        builtins._save_breeze_wav(full_audio, sr_used, full_path)
        print(f'\n  Concatenated chapter: {full_path}')
        print(f'  Total duration: {len(full_audio)/sr_used:.1f}s across {len(audio_parts)} chunks')
        results.append(str(full_path))
    else:
        print('\n  [warn] No chunks generated.')

# ═══ JSON MODE ══════════════════════════════════════════════════════════
else:
    batch_path = Path(BATCH_JSON_PATH)
    if not batch_path.exists():
        batch_path.parent.mkdir(parents=True, exist_ok=True)
        starter = [
            {'text': 'Welcome aboard. Your journey begins now.',
             'instruction': 'A warm, thoughtful young woman with a clear voice.',
             'mode': 'design', 'cfg_scale': 4.0, 'seed': 42},
            {'text': '(sigh) It is good to hear your voice again after all this time.',
             'mode': 'design', 'cfg_scale': 4.0, 'seed': 43},
            {'text': 'We need to discuss what happened last night.',
             'instruction': 'Speak slowly with a restrained, serious tone.',
             'mode': 'direction',
             'ref_audio': '/content/drive/MyDrive/ref_voice.wav',
             'ref_text': 'This is the exact transcript of the reference audio.',
             'cfg_scale': 4.0, 'seed': 7},
        ]
        batch_path.write_text(json.dumps(starter, indent=2))
        print(f'  Wrote starter batch: {batch_path}')
        print('  Edit it, then re-run STEP 7.')
    else:
        print(f'  Using existing batch JSON: {batch_path}')

    scenes = json.loads(batch_path.read_text())
    if not isinstance(scenes, list):
        raise SystemExit(f'Batch JSON must be a list, got {type(scenes).__name__}')
    print(f'  Scenes in batch: {len(scenes)}')

    for i, sc in enumerate(scenes):
        text = (sc.get('text') or '').strip()
        if not text:
            print(f'  [{i+1}/{len(scenes)}] SKIP: empty text')
            results.append(None)
            continue
        s_mode = sc.get('mode', MODE)
        s_cfg = float(sc.get('cfg_scale', 1.0 if s_mode == 'clone' else CFG_SCALE))
        s_instr = sc.get('instruction', INSTRUCTION)
        s_seed = int(sc.get('seed', SEED))
        s_ref_audio = sc.get('ref_audio', REF_AUDIO_PATH)
        s_ref_text = sc.get('ref_text', REF_TEXT)

        hash_id = _hash_key(text, s_mode, s_instr, s_cfg)
        existing = sorted(OUT_DIR.glob(f'*_seed*_{hash_id}*.wav'))
        if SKIP_EXISTING and existing:
            print(f'  [{i+1}/{len(scenes)}] SKIP (hash match): {existing[0].name}')
            results.append(str(existing[0]))
            continue

        print(f'  [{i+1}/{len(scenes)}] mode={s_mode} cfg={s_cfg} seed={s_seed}')
        print(f'    text: {text[:60]}...')
        try:
            t0 = time.time()
            kwargs = {}
            if s_ref_audio and str(s_ref_audio).strip():
                kwargs['ref_audio_path'] = str(s_ref_audio).strip()
            if s_ref_text and str(s_ref_text).strip():
                kwargs['ref_text'] = str(s_ref_text).strip()
            result = builtins._generate_speech(
                text=text,
                instruction=s_instr if s_instr else None,
                mode=s_mode,
                cfg_scale=s_cfg,
                seed=s_seed,
                save=True,
                prefix=f'batch{i+1:03d}',
                **kwargs,
            )
            elapsed = time.time() - t0
            print(f'    -> {elapsed:.0f}s, {result["duration_s"]:.1f}s audio')
            results.append(result['wav_path'])
            with _log_path.open('a') as f:
                f.write(json.dumps({'mode': 'json', 'index': i, 'status': 'ok',
                                     'path': result['wav_path'],
                                     'duration_s': result['duration_s'],
                                     'seed': result['seed_used'],
                                     'hash': hash_id}) + '\n')
        except Exception as e:
            print(f'    FAIL: {type(e).__name__}: {e}')
            results.append(None)
            with _log_path.open('a') as f:
                f.write(json.dumps({'mode': 'json', 'index': i, 'status': 'fail',
                                     'error': f'{type(e).__name__}: {e}'}) + '\n')

# ── Batch summary ───────────────────────────────────────────────────────
total = time.time() - total_start
ok_count = sum(1 for r in results if r is not None)
print('\n' + '=' * 72)
print('  BATCH SUMMARY')
print('=' * 72)
print(f'  Mode    : {BATCH_MODE}')
print(f'  Clips   : {ok_count} succeeded, {len(results) - ok_count} failed/skipped')
s = int(total); h, rem = divmod(s, 3600); m, ss = divmod(rem, 60)
print(f'  Total   : {h}:{m:02d}:{ss:02d}')
print(f'  Log     : {_log_path}')
print('=' * 72)
for r in results:
    if r:
        print(f'    {r}')


In [ ]:
#@title STEP 8 — Tail the batch log (debug aid)

"""
Prints the tail of batch_speech.log.jsonl so you can see which chunks
completed and which failed without scrolling through STEP 7's output.
"""
import json
from pathlib import Path

_log_path = builtins.Breeze_TTS_OUT_DIR / 'batch_speech.log.jsonl'
TAIL_LINES = 30  #@param {type:'slider', min:5, max:200, step:5}

if not _log_path.exists():
    print(f'No log file yet at {_log_path}')
    print('Run STEP 7 to generate speech (creates the log).')
else:
    lines = _log_path.read_text().splitlines()
    tail = lines[-TAIL_LINES:]
    print(f'Tail of {_log_path.name} ({len(lines)} total entries):\n')
    for line in tail:
        try:
            entry = json.loads(line)
            idx = entry.get('index', '?')
            st = entry.get('status', '?')
            extras = []
            if 'path' in entry:
                extras.append(f'path={Path(entry["path"]).name}')
            if 'duration_s' in entry:
                extras.append(f'duration={entry["duration_s"]:.1f}s')
            if 'seed' in entry:
                extras.append(f'seed={entry["seed"]}')
            if 'error' in entry:
                extras.append(f'error={entry["error"]}')
            extras_str = ' '.join(extras)
            print(f'  [{idx:>4}] {st:5s} {extras_str}')
        except Exception:
            print(f'  (raw) {line[:200]}')
